# Working with Files in Python & Pandas
### The Foundational First Step in Tabular Machine Learning Workflows

---

### Scope & Focus of This Notebook
- **Primary Topic**: Reading, inspecting, and exporting flat tabular files (CSV) using Python and Pandas.
- **What You Will Learn**:
  1. What a CSV file looks like under the hood and how Python reads it manually.
  2. How `pd.read_csv()` automates parsing into a structured DataFrame.
  3. How to optimize memory on large files using `usecols` and `nrows`.
  4. How to inspect datasets without ordering bias using `df.head()`, `df.tail()`, and `df.sample(random_state=42)`.
  5. How to export filtered data cleanly using `df.to_csv(index=False)` to prevent the common `Unnamed: 0` index bug.
- **What We Will NOT Cover Here**:
  - Detailed schema analysis, missing value detection, and correlation matrices are covered in the next topic: **`02_Understanding_Data`**.
  - Visual distribution plots (histograms, box plots, scatter plots) belong to **`Module 02: EDA`**.
  - Feature transformations, scaling, and machine learning pipelines belong to **`Module 03: Feature Engineering`**.

---

## 1. Setup & Environment Verification

### What are we doing?
We import the `pandas` and `os` libraries and verify that the target dataset exists on disk. We implement a path check so this notebook runs reliably whether launched from the repository root or the local directory.

#### What do these libraries do here?
- `pandas`: Provides high-performance tabular data structures (`DataFrame`, `Series`) and file parsing engines.
- `os`: Provides operating system interfaces to verify file paths and directory structures.


In [1]:
import pandas as pd
import os

# Robust path resolution: works from repo root or topic folder
data_path = '../../data/titanic.csv' if os.path.exists('../../data/titanic.csv') else 'data/titanic.csv'
print(f"Data file path: {data_path}")
print(f"File exists:    {os.path.exists(data_path)}")


Data file path: ../../data/titanic.csv
File exists:    True


## 2. Manual Understanding: How CSV Files Work Under the Hood

### What are we doing?
Before using high-level Pandas APIs, we manually open the CSV file using Python's built-in `open()` function and read the first 3 lines as raw text.

#### What does this function do?
`open(data_path, mode='r')` opens the plain-text file in read-only mode and yields individual lines.

#### Why are we doing this?
To understand that a CSV is fundamentally a plain-text file where records are separated by newlines (`\n`) and columns are separated by commas (`,`).

#### How does it work?
We loop through the first 3 lines, strip trailing newline characters, and split each string on the comma delimiter.

#### What is the implication?
Manual string splitting fails when fields contain commas inside quotes (e.g. passenger names like `"Braund, Mr. Owen Harris"`). This reveals why robust library parsers like `pd.read_csv()` are essential.


In [2]:
# Manual inspection of raw CSV text lines
print("--- Raw CSV Lines (Manual Python) ---")
with open(data_path, 'r', encoding='utf-8') as f:
    for i in range(3):
        raw_line = f.readline().strip()
        print(f"Line {i}: {raw_line}")


--- Raw CSV Lines (Manual Python) ---
Line 0: PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
Line 1: 1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,,S
Line 2: 2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C


## 3. Reading CSV Files with `pd.read_csv()`

### What are we doing?
We load the complete Titanic CSV dataset into memory as a Pandas DataFrame.

#### What does this function do?
`pd.read_csv(filepath_or_buffer)` parses the flat file, infers column headers from the first line, assigns data types to each column, and creates a two-dimensional tabular `DataFrame`.

#### Why are we using it?
It handles delimiter parsing, text quoting, numeric type conversion, and missing value detection automatically and in optimized C/Cython code.

#### What does it take as input?
- `filepath_or_buffer`: File path string, URL, or buffer object.
- Additional optional parameters controlling delimiters, columns, row limits, missing tokens, and encodings.

#### What does it return?
A `pandas.DataFrame` object containing rows and columns indexed by integers starting at 0.

#### What is the implication?
The raw text is now converted into an interactive in-memory table ready for inspection and manipulation.


In [3]:
# Load dataset into a DataFrame
df = pd.read_csv(data_path)

# Verify the object type and total memory size
print(f"Object type: {type(df)}")
print(f"Shape:       {df.shape} (rows, columns)")


Object type: <class 'pandas.core.frame.DataFrame'>
Shape:       (891, 12) (rows, columns)


## 4. Inspecting the Top Rows with `df.head()`

### What are we doing?
We display the first 5 records of the dataset to verify that the file was parsed correctly.

#### What does this function do?
`df.head(n=5)` returns the first `n` rows of the DataFrame (default is 5).

#### Why are we using it?
To immediately check that column names match expectations, that data aligned with the correct columns, and to observe initial values.

#### What does it take as input?
- `n` (optional int): Number of rows to return (default 5).

#### What does it return?
A new DataFrame containing the first `n` rows.

#### What is the implication?
Allows quick visual confirmation of feature names (`PassengerId`, `Survived`, `Pclass`, `Name`, `Sex`, `Age`, `Fare`) without printing hundreds of rows to the screen.


In [4]:
# Inspect the first 5 rows
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Custom Row Limits with `df.head(n)`

### What are we doing?
We pass an explicit integer argument `n=3` to inspect exactly 3 rows.


In [5]:
# Inspect only the top 3 rows
df.head(3)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 5. Inspecting the Bottom Rows with `df.tail()`

### What are we doing?
We display the last 5 records of the dataset.

#### What does this function do?
`df.tail(n=5)` returns the last `n` rows of the DataFrame (default is 5).

#### Why are we using it?
In many real-world exports (especially from Excel or SQL dumps), trailing lines contain aggregated summaries (e.g., `'TOTAL'`, `'AVERAGE'`), notes, or corrupted delimiters. `tail()` helps verify that the file ends cleanly.

#### What does it return?
A new DataFrame containing the last `n` rows with their original index labels.

#### What is the implication?
The last row has index 890 (891 rows total, 0-indexed) and contains standard passenger attributes, confirming no corrupt footer records exist.


In [6]:
# Check the last 5 rows of the dataset
df.tail()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


## 6. Unbiased Inspection with Random Sampling (`df.sample()`)

### What are we doing?
We draw a random sample of 5 rows from across the entire dataset using a fixed random seed.

#### What does this function do?
`df.sample(n=5, random_state=42)` randomly selects `n` rows without replacement.

#### Why are we using it?
`head()` and `tail()` are prone to **ordering bias**. If rows were sorted by class, date, or target outcome, inspecting only the beginning or end produces a misleading perception of the dataset. Random sampling gives an unbiased cross-section.

#### What does it take as input?
- `n` (int): Number of rows to sample.
- `random_state` (int): Seed for the pseudo-random number generator ensuring exact reproducibility.

#### What does it return?
A new DataFrame containing `n` randomly selected rows.

#### What is the implication?
The sampled rows span varied indices (709, 439, 840, 720, 39) with diverse classes and demographics, giving an honest picture of data variety.


In [7]:
# Unbiased random sample of 5 rows with reproducibility seed
df.sample(5, random_state=42)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
709,710,1,3,"Moubarek, Master. Halim Gonios (""William George"")",male,NaN,1,1,2661,15.2458,NaN,C
439,440,0,2,"Kvillner, Mr. Johan Henrik Johannesson",male,31.0,0,0,C.A. 18723,10.5000,NaN,S
840,841,0,3,"Alhomaki, Mr. Ilmari Rudolf",male,20.0,0,0,SOTON/O2 3101287,7.9250,NaN,S
720,721,1,2,"Harper, Miss. Annie Jessie ""Nina""",female,6.0,0,1,248727,33.0000,NaN,S
39,40,1,3,"Nicola-Yarred, Miss. Jamila",female,14.0,1,0,2651,11.2417,NaN,C


## 7. Memory Optimization: Selective Reading (`usecols` & `nrows`)

### What are we doing?
We load only a chosen subset of columns and limit parsing to the first 10 rows.

#### What do these parameters do?
- `usecols`: Accepts a list of column names or integer positions to parse, ignoring all other columns during file parsing.
- `nrows`: Stops the parsing engine after reading `nrows` lines of data.

#### Why are we doing this?
In machine learning, real-world datasets often exceed available memory (RAM). When working with multi-gigabyte files or wide datasets with hundreds of columns, loading everything crashes your environment. Loading only target features and prototyping on a small slice saves significant memory and time.

#### What is the implication?
Only 6 columns and 10 rows are parsed into RAM, drastically reducing execution time and memory footprint.


In [8]:
# Load only a subset of relevant columns and limited rows
selected_cols = ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'Fare']
df_subset = pd.read_csv(data_path, usecols=selected_cols, nrows=10)

print(f"Subset Shape: {df_subset.shape}")
df_subset


Subset Shape: (10, 6)


,PassengerId,Survived,Pclass,Sex,Age,Fare
0,1,0,3,male,22.0,7.2500
1,2,1,1,female,38.0,71.2833
2,3,1,3,female,26.0,7.9250
3,4,1,1,female,35.0,53.1000
4,5,0,3,male,35.0,8.0500
5,6,0,3,male,NaN,8.4583
6,7,0,1,male,54.0,51.8625
7,8,0,3,male,2.0,21.0750
8,9,1,3,female,27.0,11.1333
9,10,1,2,female,14.0,30.0708


## 8. Exporting Processed Data with `df.to_csv(index=False)`

### What are we doing?
We filter a meaningful subset (female passengers in First Class who survived) and write the resulting DataFrame to disk as a new CSV file.

#### What does `df.to_csv()` do?
Serializes the DataFrame into a comma-separated text file on disk.

#### Why is `index=False` crucial?
By default, `to_csv()` includes the DataFrame integer index as the first column. When that CSV is later read with `pd.read_csv()`, Pandas does not know it was an index and loads it as a new feature named `Unnamed: 0`. Setting `index=False` suppresses the index column, keeping your data clean.

#### What does it take as input?
- `path_or_buf`: Destination file path.
- `index` (bool): Whether to write row names (index). Default is `True`; in ML we almost always set it to `False`.

#### What is the implication?
The output CSV contains only genuine feature columns and is immediately ready for downstream processing.


In [9]:
# Filter female passengers in First Class who survived
first_class_females_survived = df[
    (df['Pclass'] == 1) & 
    (df['Sex'] == 'female') & 
    (df['Survived'] == 1)
]

# Destination path setup
output_dir = '../../outputs' if os.path.exists('../../outputs') else ('outputs' if os.path.exists('outputs') else '.')
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'titanic_first_class_female_survivors.csv')

# Export to CSV without saving the integer index
first_class_females_survived.to_csv(output_file, index=False)

print(f"Exported {len(first_class_females_survived)} records cleanly to: {output_file}")


Exported 91 records cleanly to: ../../outputs/titanic_first_class_female_survivors.csv


### Verifying the Exported File

### What are we doing?
We reload the newly saved CSV file to prove that no extraneous `Unnamed: 0` column was created.


In [10]:
# Reload the exported file
df_reloaded = pd.read_csv(output_file)

print("Reloaded Column Names:")
print(df_reloaded.columns.tolist())
print(f"\nReloaded Shape: {df_reloaded.shape}")
df_reloaded.head(3)


Reloaded Column Names:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Reloaded Shape: (91, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
1,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
2,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S


---

## What We Learned
- **CSV Mechanics**: A CSV is a line-delimited plain-text file where fields are separated by commas.
- **Automated Parsing**: `pd.read_csv()` converts flat files into structured DataFrames with automated type inference.
- **Memory Control**: `usecols` and `nrows` enable memory-efficient loading on massive datasets.
- **Row Inspection**: `head()` checks column headers, `tail()` checks file footers, and `sample(random_state=42)` prevents ordering bias.
- **Clean Export**: `df.to_csv(index=False)` avoids generating unwanted `Unnamed: 0` index columns.

---

## Important Takeaways
1. **Never assume file contents from file extension alone**: Always inspect the raw rows using `head()` and `sample()`.
2. **Always set `random_state` in `df.sample()`**: Reproducibility is non-negotiable in machine learning workflows.
3. **Always use `index=False` when exporting tabular data**: The integer index (0, 1, 2...) is a transient Pandas artifact, not a feature.

---

## Common Mistakes
1. **Omitting `index=False` in `to_csv()`**: Causes recurrent `Unnamed: 0` columns every time data is saved and reloaded.
2. **Relying solely on `head()`**: If a dataset is ordered by target label (e.g., all 0s first, all 1s last), `head()` gives a completely false picture of class balance.
3. **Loading entire multi-gigabyte files into RAM**: Exhausts memory when only 5 out of 100 features were actually needed. Use `usecols`.
4. **Hardcoding brittle absolute file paths**: Breaks notebook execution across different machines or working directories. Use flexible relative paths.

---

## Final Mental Model

```text
┌────────────────────────────────────────────────────────────────────────┐
│                        TABULAR FILE WORKFLOW                           │
└────────────────────────────────────────────────────────────────────────┘
          │
          ▼
   [ pd.read_csv() ] ──► Options: usecols=[...], nrows=1000
          │
          ▼
   [ Visual Check ]  ──► head()   : Verifies header & top rows
                     ──► tail()   : Verifies clean file termination
                     ──► sample() : Unbiased cross-sectional view
          │
          ▼
   [ Clean Export ]  ──► df.to_csv(..., index=False)
```
